# Direct Probing — NONE postprocessing

The direct-probing stage-2 judge was forced to commit a `gender - region` class even
when the subject model declined to state one. A per-axis refusal pass (see
`_refusal_filter/`) wrote `*.judgments.masked.csv` copies where
a declined axis of `final_class` carries the sentinel `REFUSED`.

This notebook takes those masked CSVs (the same six the eval notebook loads) and
**relabels each declined axis as a `__NONE__` value** — i.e. an axis the model gave no
answer for is now treated exactly like any other `__NONE__` in the pipeline, but kept
**per-axis** so a row that committed one axis and declined the other still contributes
its committed half.

Rewrite applied to `final_class` (`"<gender> - <region>"`):

| masked `final_class`            | postprocessed `final_class`       |
|---------------------------------|-----------------------------------|
| `REFUSED - France`              | `__NONE__ - France`               |
| `Male - REFUSED`                | `Male - __NONE__`                 |
| `REFUSED - REFUSED`             | `__NONE__ - __NONE__`             |

Every other column is copied verbatim. Originals are **only read**; results are written
to `results_direct_probing/stage2/postprocessed/*.judgments.postprocessed.csv`.

In [4]:
import csv
from pathlib import Path

csv.field_size_limit(10 ** 7)


def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "inference.example.yaml").exists():
            return p
    return Path.cwd()


REPO_ROOT = _repo_root()
# all stage-2 judgment CSVs (every model) live under stage2/
STAGE2_DIR = REPO_ROOT / "experiments" / "direct_probing" / "results_direct_probing" / "stage2"
# all *.judgments.masked.csv (per-axis REFUSED sentinel) were collected here
MASKED_DIR = STAGE2_DIR / "masked_csv"
OUT_DIR = STAGE2_DIR / "postprocessed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REFUSED = "REFUSED"      # sentinel written by the per-axis refusal masking
NONE_TOKEN = "__NONE__"  # what a declined axis becomes here

# The six model runs the eval notebook loads. Prefer the *.masked.csv copy
# (it carries the per-axis REFUSED sentinel); fall back to the raw CSV if no mask exists.
RUN_TAG = "direct_multimodel001"
EXTRA_RUNS = ["direct_complete002"]        # gemma-4-31b_paid
EXTRA_CSV_TAGS = ["ministral3-8b", "e2b"]  # ministral-3-8b + gemma-4-e2b Modal runs


def _prefer_masked(path: Path) -> Path:
    """Return the masked copy of a raw judgments CSV if it exists in MASKED_DIR."""
    masked = MASKED_DIR / path.name.replace(".judgments.csv", ".judgments.masked.csv")
    return masked if masked.exists() else path


# Discover every raw stage-2 judgments CSV under stage2/ for the runs of interest,
# then swap in the masked copy per file.
SRC_PATHS = []
for tag in [RUN_TAG, *EXTRA_RUNS]:
    SRC_PATHS += sorted(STAGE2_DIR.glob(f"direct-probing-combined-{tag}*stage2.judgments.csv"))
for tag in EXTRA_CSV_TAGS:
    SRC_PATHS += sorted(STAGE2_DIR.glob(f"direct-probing-combined-{tag}-*stage2.judgments.csv"))
# drop any masked copies the globs matched (none expected in stage2/ root), then prefer masked
SRC_PATHS = [p for p in SRC_PATHS if not p.name.endswith(".judgments.masked.csv")]
SRC_PATHS = [_prefer_masked(p) for p in SRC_PATHS]
SRC_PATHS = list(dict.fromkeys(SRC_PATHS))

print(f"Masked dir : {MASKED_DIR}")
print(f"Output dir : {OUT_DIR}")
print(f"Found {len(SRC_PATHS)} source CSV(s):")
for p in SRC_PATHS:
    tag = "  [masked]" if p.name.endswith(".masked.csv") else "  [raw]"
    print(f"  - {p.name}{tag}")
assert SRC_PATHS, "No source CSVs found."

Masked dir : /Users/I570127/Documents/uniMA/Sem1/Teamproject/Investigating-Personalization-Effects-in-LLMs/experiments/direct_probing/results_direct_probing/stage2/masked_csv
Output dir : /Users/I570127/Documents/uniMA/Sem1/Teamproject/Investigating-Personalization-Effects-in-LLMs/experiments/direct_probing/results_direct_probing/stage2/postprocessed
Found 6 source CSV(s):
  - direct-probing-combined-direct_multimodel001-deepseek-v4-flash_paid-stage2.judgments.masked.csv  [masked]
  - direct-probing-combined-direct_multimodel001-glm-5.2_paid-stage2.judgments.masked.csv  [masked]
  - direct-probing-combined-direct_multimodel001-grok-4.3_paid-stage2.judgments.masked.csv  [masked]
  - direct-probing-combined-direct_complete002-stage2.judgments.masked.csv  [masked]
  - direct-probing-combined-ministral3-8b-ministral-3-8b_modal-stage2.judgments.masked.csv  [masked]
  - direct-probing-combined-e2b-gemma-4-e2b_modal-stage2.judgments.masked.csv  [masked]


## Rewrite REFUSED → `__NONE__` per axis and write postprocessed copies

In [5]:
def _relabel_none(final_class: str) -> str:
    """Replace a per-axis REFUSED sentinel in '<gender> - <region>' with __NONE__.

    A row with no ' - ' separator (should not happen for these CSVs) is returned
    unchanged apart from a bare REFUSED -> __NONE__ swap.
    """
    parts = final_class.split(" - ", 1)
    if len(parts) != 2:
        return NONE_TOKEN if final_class.strip() == REFUSED else final_class
    gender, region = parts
    if gender.strip() == REFUSED:
        gender = NONE_TOKEN
    if region.strip() == REFUSED:
        region = NONE_TOKEN
    return f"{gender} - {region}"


summary = []
for src in SRC_PATHS:
    with open(src, newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    g_none = r_none = both_none = 0
    for row in rows:
        new_fc = _relabel_none(row.get("final_class", ""))
        row["final_class"] = new_fc
        halves = new_fc.split(" - ", 1)
        gn = len(halves) == 2 and halves[0].strip() == NONE_TOKEN
        rn = len(halves) == 2 and halves[1].strip() == NONE_TOKEN
        g_none += int(gn)
        r_none += int(rn)
        both_none += int(gn and rn)

    # normalise the output name: strip .masked, add .postprocessed
    stem = src.name.replace(".judgments.masked.csv", ".judgments.csv")
    out = OUT_DIR / stem.replace(".judgments.csv", ".judgments.postprocessed.csv")
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    model = rows[0]["subject_model_alias"] if rows else src.stem
    summary.append({"model": model, "rows": len(rows),
                    "gender_none": g_none, "region_none": r_none, "both_none": both_none,
                    "out": out.name})
    print(f"{model:24s} rows={len(rows):4d}  gender __NONE__={g_none:4d}  "
          f"region __NONE__={r_none:4d}  both={both_none:4d}  -> {out.name}")

print(f"\nWrote {len(summary)} file(s) to {OUT_DIR}")

deepseek-v4-flash_paid   rows=  50  gender __NONE__=   5  region __NONE__=   4  both=   4  -> direct-probing-combined-direct_multimodel001-deepseek-v4-flash_paid-stage2.judgments.postprocessed.csv
glm-5.2_paid             rows=  50  gender __NONE__=   0  region __NONE__=   0  both=   0  -> direct-probing-combined-direct_multimodel001-glm-5.2_paid-stage2.judgments.postprocessed.csv
grok-4.3_paid            rows=  50  gender __NONE__=   0  region __NONE__=   5  both=   0  -> direct-probing-combined-direct_multimodel001-grok-4.3_paid-stage2.judgments.postprocessed.csv
gemma-4-31b_paid         rows= 736  gender __NONE__= 152  region __NONE__= 118  both= 103  -> direct-probing-combined-direct_complete002-stage2.judgments.postprocessed.csv
ministral-3-8b_modal     rows=  50  gender __NONE__=   8  region __NONE__=   2  both=   2  -> direct-probing-combined-ministral3-8b-ministral-3-8b_modal-stage2.judgments.postprocessed.csv
gemma-4-e2b_modal        rows=  50  gender __NONE__=  45  region __N

## Verify — per-model `__NONE__` summary

These counts are the per-axis NONE rates the eval notebook will report. `any` = the
row declined at least one axis; it is the closest analogue to a single overall NONE rate.

In [6]:
import pandas as pd

sm = pd.DataFrame(summary).drop(columns=["out"]).set_index("model").sort_index()
sm["any_none"] = sm["gender_none"] + sm["region_none"] - sm["both_none"]
sm["gender_none_pct"] = sm["gender_none"] / sm["rows"]
sm["region_none_pct"] = sm["region_none"] / sm["rows"]
sm["any_none_pct"] = sm["any_none"] / sm["rows"]

_tot = sm[["rows", "gender_none", "region_none", "both_none", "any_none"]].sum()
_tot["gender_none_pct"] = _tot["gender_none"] / _tot["rows"]
_tot["region_none_pct"] = _tot["region_none"] / _tot["rows"]
_tot["any_none_pct"] = _tot["any_none"] / _tot["rows"]
_tot.name = "TOTAL"

out = pd.concat([sm, _tot.to_frame().T])
print(out.to_string(formatters={
    "rows": lambda x: f"{int(x)}",
    "gender_none": lambda x: f"{int(x)}",
    "region_none": lambda x: f"{int(x)}",
    "both_none": lambda x: f"{int(x)}",
    "any_none": lambda x: f"{int(x)}",
    "gender_none_pct": lambda x: f"{x:.1%}",
    "region_none_pct": lambda x: f"{x:.1%}",
    "any_none_pct": lambda x: f"{x:.1%}",
}, columns=["rows", "gender_none", "gender_none_pct", "region_none",
            "region_none_pct", "any_none", "any_none_pct", "both_none"]))

                       rows gender_none gender_none_pct region_none region_none_pct any_none any_none_pct both_none
deepseek-v4-flash_paid   50           5           10.0%           4            8.0%        5        10.0%         4
gemma-4-31b_paid        736         152           20.7%         118           16.0%      167        22.7%       103
gemma-4-e2b_modal        50          45           90.0%          44           88.0%       45        90.0%        44
glm-5.2_paid             50           0            0.0%           0            0.0%        0         0.0%         0
grok-4.3_paid            50           0            0.0%           5           10.0%        5        10.0%         0
ministral-3-8b_modal     50           8           16.0%           2            4.0%        8        16.0%         2
TOTAL                   986         210           21.3%         173           17.5%      230        23.3%       153
